데이터와 모델 모두 최소한으로 가공해서 학습한다. 

예측을 내고 캐글에 제출까지 해본다.

### 데이터 타입 범위 표

<img src = https://user-images.githubusercontent.com/89520805/166191804-c083cbd2-5f3f-4850-8f4f-41fc8982a090.png height = 500 width = 550>

# 1. 데이터 불러오기

In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.model_selection import train_test_split

path = '/content/drive/MyDrive/data/instacart-market-basket-analysis/' # csv파일이 들어있는 폴더 경로 지정

## 데이터 타입 변환

데이터 타입에 따라서 메모리를 차지하는 용량이 차이가 난다.

판다스 데이터 프레임은 기본적으로 가장 큰 범위인 int64를 사용한다.
따라서 데이터에 따라서 더 작은 범위의 데이터타입을 사용하면 메모리를 훨씬 효율적으로
사용할 수 있다. 

In [4]:
priors = pd.read_csv(path + 'order_products__prior.csv', dtype={
            'order_id': np.uint32,          # 1 ~ 3421083
            'product_id': np.uint16,        # 1 ~ 49688
            'add_to_cart_order': np.uint8,  # 1 ~ 80 
            'reordered': np.uint8})         # 0 ~ 1
train = pd.read_csv(path + 'order_products__train.csv', dtype={
            'order_id': np.uint32,          # 1 ~ 3421083
            'product_id': np.uint16,        # 1 ~ 49688
            'add_to_cart_order': np.uint8,   # 1 ~ 80
            'reordered': np.uint8})         # 0 ~ 1
orders = pd.read_csv(path + 'orders.csv', dtype={
        'order_id': np.uint32,              # 1 ~ 3421083
        'user_id': np.uint32,               # 1 ~ 206209
        'eval_set': 'category',             
        'order_number': np.uint8,           # 1 ~ 100
        'order_dow': np.uint8,              # 0 ~ 6 
        'order_hour_of_day': np.uint8,      # 0 ~ 23
        'days_since_prior_order': np.float32})
products = pd.read_csv(path + 'products.csv', dtype={
        'product_id': np.uint16,    # 1 ~ 49688
        'order_id': np.uint32,
        'aisle_id': np.uint8,       # 1 ~ 134
        'department_id': np.uint8}, # 1 ~ 21
        usecols=['product_id', 'aisle_id', 'department_id'])

# 2. 데이터 전처리 하기

데이터를 모델에 넣을 수 있게 X_train, y_train, X_test, y_test 형식으로 변환해야 한다. 

먼저 orders DF를 살펴보자. 20만 유저의 340만 건의 주문이 있다. 

유저별로 최소 4개에서 100개의 주문 내역이 있다. 모든 유저의 마지막 주문은 train과  test로 구분되어 있다. 모든 유저의 마지막 주문이니까 그 수는 모든 유저의 수와 같다. 13만 개의 주문은 train셋, 7.5만 개의 주문은 test셋이다. train셋과 test셋의 차이는 train데이터는 해당 주문별로 유저가 어떤 제품을 구매했는지 알 수 있다. 

따라서 우리는 train 데이터의 주문 번호와 구매한 제품을 가지고 훈련을 해서 test데이터의 주문번호 마다 어떤 제품이 들어있을지 예측해야 한다. 

제출 형식은 컬럼1이 test데이터의 order_id이고 컬럼2가 product_id를 공백으로 구분해서 넣는다. 만약 유저가 구매하지 않았다면 `None`을 넣는다. 

## train셋

train 데이터는 feature와 label로 나눠야한다. 일단 필요한 컬럼은 아래와 같다.

### feature

1. order_id 
2.해당 유저가 구매한 모든 제품_id 
3.제품이나 주문, 고객에 대한 feature들.

### label

- 실제로 구매한 제품은 1, 그렇지 않으면 0. 

In [5]:
# priors에 orders 추가 -> 유저, 주문, 제품 정보를 한 번에 보기 위해서
orders.set_index('order_id', inplace=True, drop=False)
priors = priors.join(orders, on='order_id', rsuffix='_')    # 뒤에 '_' 붙이기
priors.drop('order_id_', inplace=True, axis=1)

# 유저가 구매한 모든 상품 DF 생성
users = pd.DataFrame()
users['all_products'] = priors.groupby('user_id')['product_id'].apply(set)

# order 분리하기
train_orders = orders[orders['eval_set'] == 'train'].drop('eval_set', axis=1)
test_orders = orders[orders['eval_set'] == 'test'].drop('eval_set', axis=1)

train.set_index(['order_id', 'product_id'], inplace=True, drop=False)

In [6]:
def features(selected_orders, labels_given=False):
    order_list = []
    product_list = []
    labels = []

    train_index = set(train.index) # order_id마다 구매한 모든 제품과 매칭되어 반환
    """
        (1, 10246),
        (1, 11109),
        (1, 13176)
    """
    for row in tqdm(selected_orders.to_numpy()):    # orders를 한 행씩 꺼내온다.numpy array로 가져오면 속도가 빠르다.
        order_id = row[0].astype(np.int32)          
        user_id = row[1].astype(np.int32)
        user_prods = users['all_products'][user_id] # 해당 유저가 구매한 모든 제품 
        product_list += user_prods                 # 유저가 구매한 모든 제품을 담는다.
        order_list += [order_id] * len(user_prods) # 유저가 구매한 모든 제품의 개수만큼 order_id를 복사한다.  
        if labels_given:                           
            # 유저가 구매한 모든 제품을 꺼내서 order_id와 묶는다. 그리고 실제로 해당 주문에 이 제품을 구매했는지 비교해본다. True or False로 값이 담긴다. 
            labels += [(order_id, prod) in train_index for prod in user_prods]  
        
    df = pd.DataFrame({'order_id':order_list, 'product_id':product_list}, dtype=np.int32)
    labels = np.array(labels, dtype=np.int8)    # 타겟을 정수로 변환
    del order_list, product_list

    """
    여기에 feature 추가하기
    """

    return (df, labels)

In [7]:
X, y = features(train_orders, labels_given=True) # X, y 생성 
X_test, _ = features(test_orders)               # test 셋 생성
X.shape, y.shape, X_test.shape

100%|██████████| 75000/75000 [00:01<00:00, 57290.07it/s]


((8474661, 2), (8474661,), (4833292, 2))

In [25]:
# 훈련셋의 레이블의 분포를 비교해보면 0인 레이블이 90% 이상이다.
print(f"label 0 : {y.tolist().count(0)} label 1 : {y.tolist().count(1)}")
print(f"label 0 비율: {y.tolist().count(0)/y.shape[0]:.3f} label 1 비율: {y.tolist().count(1)/y.shape[0]:.3f}")

label 0 : 7645837 label 1 : 828824
label 0 비율: 0.902 label 1 비율: 0.098


In [ ]:
# 검증셋 나누기
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
X_train.shape, y_train.shape, X_val.shape, y_val.shape

# 모델 학습, 평가

XGBoost와 lightGBM 사용하기

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier, plot_importance
from lightgbm import LGBMClassifier, plot_importance

In [ ]:
xgbc = XGBClassifier()
xgbc.fit(X_train, y_train)

preds = xgbc.predict(df_test)
df_test['pred'] = preds

In [ ]:
lgbm = LGBMClassifier()
lgbm.fit(X_train, y_train)

preds = lgbm.predict(df_test)
df_test['pred'] = preds

## 파라미터 튜닝

## 성능평가하기

- confusion matrix, f1 score, roc_auc_score

- feature importance

## Submit

### 예측 결과를 submit 형식에 맞게 변경하고 csv로 저장

In [ ]:
TRESHOLD = 0.22

d = dict()
for row in X_test.itertuples():
    if row.pred > TRESHOLD:
        try:
            d[row.order_id] += ' ' + str(row.product_id)
        except:
            d[row.order_id] = str(row.product_id)

for order in test_orders.order_id:
    if order not in d:
        d[order] = 'None'

sub = pd.DataFrame.from_dict(d, orient='index')

sub.reset_index(inplace=True)
sub.columns = ['order_id', 'products']
sub.to_csv('sub.csv', index=False)

### 캐글 API 키를 사용해서 바로 submit하기

In [ ]:
# 캐글 API 키 설정 사용하기
!mkdir -p ~/.kaggle
%cd /content/drive/MyDrive/
!chmod 600 kaggle.json
!cp kaggle.json ~/.kaggle
%cd /content

# submit코드
!kaggle competitions submit -c instacart-market-basket-analysis -f sub.csv -m "2"